#### MILE -- automatische Warmup-Adaption, ERWEITERTE Config (mehr MCMC-Budget)

Gleiche 128 Posterior-Beispiele, gleicher Prior (`prior_std=0.02`), gleicher `theta_map` wie in `02_mile_warmup.ipynb` -- nur mit deutlich mehr MCMC-Rechenbudget auf derselben Posterior (nicht mehr Daten!), um die diagnostizierte niedrige Exploration (sehr kleine Mutual Information) gezielt anzugehen:

- `WARMUP_STEPS`: 150 -> 250
- `N_SAMPLES`: 10 -> 15
- `THINNING_STEPS`: 20 -> 30
- `N_PER_CLASS`: 32 -> 128 (128 Beispiele -> 512 Beispiele) -- konsistent mit MFVI/Laplace hochgesetzt, damit der Methodenvergleich fair bleibt.

`EVAL_ON_TEST_SET = True` ist hier direkt von Anfang an gesetzt (nicht erst Dev-Subset, dann Testset) -- die Pipeline ist durch den vorherigen erfolgreichen adaptiven Lauf bereits validiert, ein zweiter teurer Sampling-Durchlauf nur fuers Dev-Set waere reine Zeitverschwendung. Ganz unten gibt es trotzdem einen optionalen Extra-Block, der zusaetzlich auf dem 128er-Dev-Subset auswertet (fuer den direkten Vorher-Nachher-Vergleich der Mutual Information mit dem alten Lauf) -- der laeuft mit den bereits gezogenen Samples, kostet also fast nichts, solange der Kernel noch lebt.


In [ ]:
from pathlib import Path

# ============================================================
# CONFIG
# MILE -- AUTOMATISCHE WARMUP-ADAPTION
# ============================================================

MASTER_DIR = Path(
    "/dss/dsshome1/00/ra58vit2/Masterarbeit"
)

QWEN_REPO = MASTER_DIR / "bayes_sub_inf"
MILE_REPO = MASTER_DIR / "MILE"

BASELINE_SCRIPT = (
    QWEN_REPO
    / "experiments"
    / "ag_news_qwen_lora"
    / "evaluate_saved_baseline.py"
)

RESULT_DIR = (
    MASTER_DIR
    / "method_results"
    / "mile_qwen_agnews"
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 2
POSTERIOR_KEY_SEED = 2027

# ------------------------------------------------------------
# Posterior data (gleiche Groessenordnung wie 02_mile.ipynb)
# ------------------------------------------------------------

N_PER_CLASS = 128        # erweitert von 32 -- 128 x 4 = 512 examples
SEQ_LEN = 32

# ------------------------------------------------------------
# Prior (gleich wie ueberall -- 0.02, nicht mehr der alte Default 1.0)
# ------------------------------------------------------------

PRIOR_STD = 0.02

# ------------------------------------------------------------
# MCLMC -- automatische Adaption
# ------------------------------------------------------------

N_CHAINS = 1

WARMUP_STEPS = 250      # erweitert von 150, aber moderater als der erste
                        # Entwurf (500) -- da N_PER_CLASS jetzt zusaetzlich
                        # 4x hoch ist, wuerden sich beide Erhoehungen sonst
                        # multiplizieren (~23x statt ~8x Gesamtkosten)
N_SAMPLES = 15           # erweitert von 10, moderater als 30 -- aus
                         # demselben Kostenabwaegungsgrund wie oben
N_THINNING = 1   # ungenutzt vom automatischen Sampler selbst -- wir thinnen manuell danach

# Gezielt False -- die Varianzschaetzung fuer diagonales Preconditioning kam
# aus einer kaum bewegten fruehen Phase (step_size_init sehr klein) und war
# die wahrscheinlichste Ursache der beiden vorherigen Divergenzen.
DIAGONAL_PRECONDITIONING = False

DESIRED_ENERGY_VAR_START = 0.0005
DESIRED_ENERGY_VAR_END = 0.0001

TRUST_IN_ESTIMATE = 1.5

NUM_EFFECTIVE_SAMPLES = 10

STEP_SIZE_INIT = 1e-5

# ------------------------------------------------------------
# Sampling-Phase: Thinning (gleicher Fix wie in 02_mile.ipynb)
# ------------------------------------------------------------

THINNING_STEPS = 30      # erweitert von 20, moderater als 50 -- aus
                         # demselben Kostenabwaegungsgrund wie oben

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

EVAL_BATCH_SIZE = 64
EVAL_ON_TEST_SET = True   # direkt auf dem echten Testset -- Pipeline
                          # ist durch den vorherigen Lauf bereits validiert

# ------------------------------------------------------------
# Automatic run name
# ------------------------------------------------------------

N_POSTERIOR_CONFIG = 4 * N_PER_CLASS

RUN_NAME = (
    f"mile_adaptive_balanced{N_POSTERIOR_CONFIG}"
    f"_seq{SEQ_LEN}"
    f"_prior{PRIOR_STD}"
    f"_warmup{WARMUP_STEPS}"
    f"_samples{N_SAMPLES}"
    f"_thin{THINNING_STEPS}"
)

print("======================================")
print("MILE -- AUTOMATIC WARMUP ADAPTION")
print("======================================")
print("Run name:              ", RUN_NAME)
print("Examples/class:        ", N_PER_CLASS)
print("Total examples:        ", N_POSTERIOR_CONFIG)
print("Prior std:             ", PRIOR_STD)
print("Warmup steps:          ", WARMUP_STEPS)
print("Samples:               ", N_SAMPLES)
print("Thinning steps/sample: ", THINNING_STEPS)
print("Diagonal precond.:     ", DIAGONAL_PRECONDITIONING)
print("Step size init:        ", STEP_SIZE_INIT)
print("======================================")


In [ ]:
# ============================================================
# IMPORTS AND ENVIRONMENT
# ============================================================

import os
import sys
import gc
import copy
import json
import runpy
import importlib

import numpy as np
import jax
import jax.numpy as jnp

from jax import random
from jax.flatten_util import ravel_pytree


RESULT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(QWEN_REPO)

if str(MILE_REPO) not in sys.path:
    sys.path.insert(0, str(MILE_REPO))

if str(QWEN_REPO) not in sys.path:
    sys.path.insert(0, str(QWEN_REPO))

print("Python:", sys.executable)
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

print("\nPaths:")
print("Qwen repo:", QWEN_REPO.exists())
print("MILE repo:", MILE_REPO.exists())
print("Baseline script:", BASELINE_SCRIPT.exists())
print("Result directory:", RESULT_DIR)

assert QWEN_REPO.exists()
assert MILE_REPO.exists()
assert BASELINE_SCRIPT.exists()


In [ ]:
# ============================================================
# GETEILTES MODUL
# ============================================================

import qwen_posterior_utils as qpu

qpu.patch_subspace_curve_predict()


In [ ]:
# ============================================================
# LOAD QWEN AG-NEWS BASELINE
# ============================================================

baseline_objects = runpy.run_path(str(BASELINE_SCRIPT))

env = baseline_objects["env"]
params = baseline_objects["params"]
data = baseline_objects["data"]
rng_key = baseline_objects["rng_key"]

val_metrics = baseline_objects["val_metrics"]
test_metrics = baseline_objects["test_metrics"]

print("\nBaseline loaded")
print("Model:", type(env.s_model))

print("\nBaseline test metrics:")
for key, value in test_metrics.items():
    print(f"{key}: {float(value):.6f}")


In [ ]:
# ============================================================
# POSTERIOR SETUP (Subset, NLL, Vektorisierung, Log-Posterior)
# ============================================================

posterior = qpu.setup_qwen_posterior(
    env=env,
    params=params,
    data=data,
    n_per_class=N_PER_CLASS,
    seq_len=SEQ_LEN,
    posterior_key_seed=POSTERIOR_KEY_SEED,
    prior_std=PRIOR_STD,
)

x_posterior = posterior.x_posterior
y_posterior = posterior.y_posterior
posterior_example_keys = posterior.posterior_example_keys
N_POSTERIOR_EXAMPLES = posterior.n_posterior_examples
subset_indices_host = posterior.subset_indices_host

theta_map = posterior.theta_map
rebuild_full_params = posterior.rebuild_full_params
qwen_log_posterior = posterior.qwen_log_posterior


In [ ]:
# ============================================================
# FULL POSTERIOR GRADIENT CHECK
# ============================================================

print("======================================")
print("FULL POSTERIOR GRADIENT CHECK")
print("======================================")
print("Posterior examples:", N_POSTERIOR_EXAMPLES)
print("Theta shape:", theta_map.shape)
print("Theta dtype:", theta_map.dtype)

print()
print("Computing full posterior gradient...")

log_post_value, log_post_gradient = (
    jax.value_and_grad(qwen_log_posterior)(theta_map)
)

log_post_value = jax.block_until_ready(log_post_value)
log_post_gradient = jax.block_until_ready(log_post_gradient)

log_post_value_host = float(jax.device_get(log_post_value))
gradient_host = np.asarray(jax.device_get(log_post_gradient))

log_post_finite = bool(np.isfinite(log_post_value_host))
gradient_finite = bool(np.isfinite(gradient_host).all())

print()
print("Log posterior:", log_post_value_host)
print("Log posterior finite:", log_post_finite)
print("Gradient finite:", gradient_finite)
print("Gradient norm:", float(np.linalg.norm(gradient_host)))
print("Gradient max abs:", float(np.max(np.abs(gradient_host))))

assert log_post_finite, "Log posterior is not finite."
assert gradient_finite, "Full posterior gradient contains NaN or Inf."

print()
print("SUCCESS: Full posterior and gradient are finite.")


### Automatische MCLMC-Warmup-Adaption (dritter Versuch)


In [ ]:
# ============================================================
# MCLMC CONFIGURATION
# ============================================================

from src.config.sampler import (
    SamplerConfig,
    Sampler,
)

from src.training.sampling import (
    warmup_mclmc,
)


sampler_config = SamplerConfig(
    name=Sampler.MCLMC,
    n_chains=N_CHAINS,
    warmup_steps=WARMUP_STEPS,
    n_samples=N_SAMPLES,
    n_thinning=N_THINNING,
    diagonal_preconditioning=DIAGONAL_PRECONDITIONING,
    desired_energy_var_start=DESIRED_ENERGY_VAR_START,
    desired_energy_var_end=DESIRED_ENERGY_VAR_END,
    trust_in_estimate=TRUST_IN_ESTIMATE,
    num_effective_samples=NUM_EFFECTIVE_SAMPLES,
    step_size_init=STEP_SIZE_INIT,
    keep_warmup=False,
)

print("Run:", RUN_NAME)
print(sampler_config)


In [ ]:
# ============================================================
# AUTOMATIC MCLMC WARMUP
# ============================================================

rng_key, warmup_key, sampling_key = random.split(rng_key, 3)

warmup_state, tuned_parameters = warmup_mclmc(
    config=sampler_config,
    rng_key=warmup_key,
    init_params=theta_map,
    unnorm_log_posterior=qwen_log_posterior,
    n_devices=1,
)

warmup_state = jax.block_until_ready(warmup_state)

print("Warmup call completed")
print("Tuned parameters:")
for name, value in tuned_parameters.items():
    value_host = np.asarray(jax.device_get(value))
    print(f"  {name}: {value_host}")


In [ ]:
# ============================================================
# VALIDATE WARMUP
# ============================================================

warmup_position_host = np.asarray(jax.device_get(warmup_state.position))
warmup_is_finite = bool(np.isfinite(warmup_position_host).all())

print("Warmup position finite:", warmup_is_finite)
print("NaN count:", int(np.isnan(warmup_position_host).sum()))
print("Inf count:", int(np.isinf(warmup_position_host).sum()))

for name, value in tuned_parameters.items():
    value_host = np.asarray(jax.device_get(value))
    print(name, "finite=", bool(np.isfinite(value_host).all()))

if warmup_is_finite:
    warmup_log_posterior = qwen_log_posterior(warmup_state.position)
    print("\nWarmup log posterior:", float(warmup_log_posterior))
    print("Warmup posterior finite:", bool(jnp.isfinite(warmup_log_posterior)))

assert warmup_is_finite, (
    "Warmup contains NaN or Inf. Do not start sampling. "
    "Fallback: 02_mile.ipynb (feste step_size) hat bereits ein gespeichertes Ergebnis."
)


In [ ]:
# ============================================================
# MCLMC SAMPLING (mit Thinning, adaptiv getunte step_size/L)
# ============================================================

mclmc_sampler = sampler_config.kernel(
    qwen_log_posterior,
    **tuned_parameters,
)


def one_thinned_sample(state, keys):
    def inner_step(inner_state, key):
        new_state, info = mclmc_sampler.step(key, inner_state)
        return new_state, None

    state, _ = jax.lax.scan(inner_step, state, keys)
    return state, state.position


rng_key, sampling_key = random.split(rng_key)
sampling_keys = random.split(
    sampling_key, N_SAMPLES * THINNING_STEPS,
).reshape(N_SAMPLES, THINNING_STEPS, -1)

final_state, sample_positions = jax.lax.scan(
    one_thinned_sample,
    warmup_state,
    sampling_keys,
)

final_state = jax.block_until_ready(final_state)
sample_positions = jax.block_until_ready(sample_positions)

samples_host = np.asarray(jax.device_get(sample_positions))
finite_per_sample = np.isfinite(samples_host).all(axis=1)

print("Sampling completed (with thinning)")
print("Thinning steps between samples:", THINNING_STEPS)
print("Samples shape:", samples_host.shape)
print("All samples finite:", finite_per_sample.all())

consecutive_distances = np.linalg.norm(np.diff(samples_host, axis=0), axis=1)
print("Distanz zwischen aufeinanderfolgenden Samples (min/mean/max):")
print(consecutive_distances.min(), consecutive_distances.mean(), consecutive_distances.max())

assert finite_per_sample.all(), "At least one sample contains NaN or Inf."


In [ ]:
# ============================================================
# BASIC SAMPLE DIAGNOSTICS
# ============================================================

distances_from_map = qpu.compute_distances_from_map(theta_map, sample_positions)


In [ ]:
# ============================================================
# EVALUATION DATA (Posterior-Subset oder echtes Testset)
# ============================================================

if EVAL_ON_TEST_SET:
    x_eval, y_eval = data.get("test")
    print("Evaluating on the full AG News TEST set.")
else:
    x_eval, y_eval = x_posterior, y_posterior
    print("Evaluating on the posterior subset (dev/stability check).")

print("Evaluation examples:", int(y_eval.shape[0]))


In [ ]:
# ============================================================
# POSTERIOR PREDICTIVE PROBABILITIES
# ============================================================

sample_probabilities, mean_probabilities, rng_key = (
    qpu.compute_posterior_predictive_probabilities(
        env=env,
        rebuild_full_params=rebuild_full_params,
        sample_thetas=samples_host,
        x_eval=x_eval,
        y_eval=y_eval,
        rng_key=rng_key,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
)


In [ ]:
# ============================================================
# METRICS
# ============================================================

metrics = qpu.compute_predictive_metrics(
    sample_probabilities=sample_probabilities,
    mean_probabilities=mean_probabilities,
    y_true=y_eval,
)


In [ ]:
# ============================================================
# EXPECTED CALIBRATION ERROR
# ============================================================

ece = qpu.multiclass_ece(mean_probabilities, y_eval, n_bins=15)
print("ECE:", float(ece))


In [ ]:
# ============================================================
# LOG POSTERIOR VALUES FOR ALL SAMPLES
# ============================================================

sample_log_posteriors_host = qpu.compute_sample_log_posteriors(
    qwen_log_posterior,
    sample_positions,
)


In [ ]:
# ============================================================
# SAVE RUN
# ============================================================

summary, metadata_arrays = qpu.summarize_metrics(metrics)

summary["ece"] = float(ece)

tuned_step_size = float(np.asarray(jax.device_get(tuned_parameters["step_size"])))
tuned_L = float(np.asarray(jax.device_get(tuned_parameters["L"])))

summary.update({
    "run_name": RUN_NAME,
    "method": "MILE",
    "sampler": "MCLMC",
    "dataset": "AG News",
    "model": "Qwen2.5-0.5B",

    "seed": SEED,
    "posterior_key_seed": POSTERIOR_KEY_SEED,

    "n_per_class": N_PER_CLASS,
    "n_posterior_examples": N_POSTERIOR_EXAMPLES,
    "sequence_length": SEQ_LEN,

    "eval_on_test_set": EVAL_ON_TEST_SET,
    "n_eval_examples": int(y_eval.shape[0]),

    "prior_std": PRIOR_STD,

    "n_chains": N_CHAINS,
    "warmup_steps": WARMUP_STEPS,
    "n_samples": N_SAMPLES,
    "thinning_steps": THINNING_STEPS,
    "diagonal_preconditioning": DIAGONAL_PRECONDITIONING,
    "desired_energy_var_start": DESIRED_ENERGY_VAR_START,
    "desired_energy_var_end": DESIRED_ENERGY_VAR_END,
    "trust_in_estimate": TRUST_IN_ESTIMATE,
    "num_effective_samples": NUM_EFFECTIVE_SAMPLES,
    "step_size_init": STEP_SIZE_INIT,
    "adaptive_warmup": True,

    "tuned_step_size": tuned_step_size,
    "tuned_L": tuned_L,

    "all_samples_finite": bool(np.isfinite(samples_host).all()),
    "warmup_position_finite": bool(np.isfinite(warmup_position_host).all()),
    "n_parameter_dimensions": int(samples_host.shape[-1]),

    "minimum_distance_from_map": float(distances_from_map.min()),
    "mean_distance_from_map": float(distances_from_map.mean()),
    "maximum_distance_from_map": float(distances_from_map.max()),

    "minimum_log_posterior": float(sample_log_posteriors_host.min()),
    "mean_log_posterior": float(sample_log_posteriors_host.mean()),
    "maximum_log_posterior": float(sample_log_posteriors_host.max()),
})

metadata_arrays.update({
    "subset_indices": np.asarray(subset_indices_host),
    "labels": np.asarray(jax.device_get(y_eval)),
    "distances_from_map": np.asarray(distances_from_map),
    "log_posteriors": sample_log_posteriors_host,
})

paths = qpu.save_method_run(
    result_dir=RESULT_DIR,
    run_name=RUN_NAME,
    sample_positions=sample_positions,
    sample_probabilities=sample_probabilities,
    metadata_arrays=metadata_arrays,
    summary=summary,
)

print()
print("Results:")
print("Accuracy:           ", summary["accuracy"])
print("LPPD:               ", summary["lppd"])
print("NLL:                ", summary["posterior_predictive_nll"])
print("Brier Score:        ", summary["brier_score"])
print("ECE:                ", summary["ece"])
print("Mean pred. entropy: ", summary["mean_predictive_entropy"])
print("Mean exp. entropy:  ", summary["mean_expected_entropy"])
print("Mean MI:            ", summary["mean_mutual_information"])
print()
print("Zum Vergleich: 02_mile.ipynb (feste step_size) Ergebnis liegt bereits in")
print(RESULT_DIR, "unter einem anderen RUN_NAME.")


### EXTRA (optional): zusaetzliche Auswertung auf dem 128er-Dev-Subset

Nur ausfuehren, wenn der Kernel nach dem Hauptlauf oben noch lebt (`sample_positions`, `theta_map`, `env`, `rebuild_full_params` etc. noch im Speicher). Nutzt exakt dieselben bereits gezogenen Samples, wertet sie nur zusaetzlich auf dem kleinen 128er-Posterior-Subset aus statt auf dem Testset -- kostet praktisch nichts (kein neues Sampling), erlaubt aber den direkten Vergleich der Mutual Information mit dem alten adaptiven Lauf (der nur auf dem Dev-Subset ausgewertet wurde). Speichert unter einem eigenen `run_name` (Suffix `_deveval`), ueberschreibt also nicht das Testset-Ergebnis von oben.


In [ ]:
# ============================================================
# EXTRA: OPTIONAL DEV-SET EVALUATION (gleiche Samples, anderes Eval-Set)
# Nur ausfuehren, wenn der Kernel nach dem Hauptlauf oben noch lebt.
# ============================================================

x_eval_dev, y_eval_dev = x_posterior, y_posterior

(
    dev_sample_probabilities,
    dev_mean_probabilities,
    rng_key,
) = qpu.compute_posterior_predictive_probabilities(
    env=env,
    rebuild_full_params=rebuild_full_params,
    sample_thetas=samples_host,
    x_eval=x_eval_dev,
    y_eval=y_eval_dev,
    rng_key=rng_key,
    eval_batch_size=EVAL_BATCH_SIZE,
)

dev_metrics = qpu.compute_predictive_metrics(
    sample_probabilities=dev_sample_probabilities,
    mean_probabilities=dev_mean_probabilities,
    y_true=y_eval_dev,
)

dev_ece = qpu.multiclass_ece(dev_mean_probabilities, y_eval_dev, n_bins=15)

dev_summary, dev_metadata_arrays = qpu.summarize_metrics(dev_metrics)
dev_summary["ece"] = float(dev_ece)

DEV_RUN_NAME = RUN_NAME + "_deveval"

dev_summary.update({
    "run_name": DEV_RUN_NAME,
    "method": "MILE",
    "sampler": "MCLMC",
    "dataset": "AG News",
    "model": "Qwen2.5-0.5B",
    "seed": SEED,
    "posterior_key_seed": POSTERIOR_KEY_SEED,
    "n_per_class": N_PER_CLASS,
    "n_posterior_examples": N_POSTERIOR_EXAMPLES,
    "sequence_length": SEQ_LEN,
    "eval_on_test_set": False,
    "n_eval_examples": int(y_eval_dev.shape[0]),
    "prior_std": PRIOR_STD,
    "warmup_steps": WARMUP_STEPS,
    "n_samples": N_SAMPLES,
    "thinning_steps": THINNING_STEPS,
    "diagonal_preconditioning": DIAGONAL_PRECONDITIONING,
    "adaptive_warmup": True,
    "tuned_step_size": tuned_step_size,
    "tuned_L": tuned_L,
    "note": (
        "Gleiche Samples wie der Testset-Lauf oben, nur anderes Eval-Set "
        "(128er-Dev-Subset) -- fuer direkten Vergleich mit dem alten "
        "adaptiven Lauf (02_mile_warmup.ipynb, dev-only)."
    ),
})

dev_metadata_arrays.update({
    "subset_indices": np.asarray(subset_indices_host),
    "labels": np.asarray(jax.device_get(y_eval_dev)),
    "distances_from_map": np.asarray(distances_from_map),
    "log_posteriors": sample_log_posteriors_host,
})

dev_paths = qpu.save_method_run(
    result_dir=RESULT_DIR,
    run_name=DEV_RUN_NAME,
    sample_positions=sample_positions,
    sample_probabilities=dev_sample_probabilities,
    metadata_arrays=dev_metadata_arrays,
    summary=dev_summary,
)

print("Dev-Set-Auswertung gespeichert unter:", DEV_RUN_NAME)
print("Accuracy:", dev_summary["accuracy"])
print("LPPD:    ", dev_summary["lppd"])
print("NLL:     ", dev_summary["posterior_predictive_nll"])
print("ECE:     ", dev_summary["ece"])
print("Mean MI: ", dev_summary["mean_mutual_information"])
